In [1]:
import os
from torch.utils.data import Dataset,DataLoader
from torchvision import transforms
from PIL import Image

In [2]:
class imageProcessor :
    def __init__(self,root_dir_path, transformations = None):
        self.root_dir_path = root_dir_path
        self.transformations = transformations

        self.all_img_paths = [os.path.join(root_dir_path, img) for img in os.listdir(root_dir_path)]

    def __len__(self):
        return len(self.all_img_paths)
    
    def __getitem__(self,idx):
        img_path = self.all_img_paths[idx]
        img = Image.open(img_path).convert("RGB")

        if self.transformations:
            img = transformations(img)
            return img

In [3]:
root_dir_path = "./img_align_celeba"

transformations = transforms.Compose([
    transforms.CenterCrop(178),#178x218 => 178x178
    transforms.Resize(64), #64x64
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)) # pixel range : [-1,1]
])

In [4]:
dataset = imageProcessor(root_dir_path, transformations)
print(f"loaded{len(dataset)}images")

loaded202600images


In [5]:
dataLoader = DataLoader(dataset, batch_size=128, shuffle=True)

## Generator network

In [6]:
import torch.nn as nn 
import torch.optim as optim
import numpy as np

In [7]:
class Generator(nn.Module):
    def __init__(self,z_dim=100, channels=3):
        super(Generator,self).__init__()

        self.model = nn.Sequential(
            nn.Linear(z_dim,256), # 100 => 256
            nn.ReLU(),

            nn.Linear(256,512),
            nn.ReLU(),

            nn.Linear(512,1024),
            nn.ReLU(),

            nn.Linear(1024, 64*64*channels),
            nn.Tanh() #[1,-1]
        )
    
    def forward(self, z):
        img = self.model(z)
        img = img.view(img.size(0),3,64,64)
        return img


## Discriminator Network

In [8]:
class Discriminator(nn.Module):
    def __init__(self, channels=3):
        super(Discriminator,self).__init__()

        self.model = nn.Sequential(
            nn.Flatten(),

            nn.Linear(64*64*3,1024), # 100 => 256
            nn.LeakyReLU(),

            nn.Linear(1024,512),
            nn.LeakyReLU(),

            nn.Linear(512,256),
            nn.LeakyReLU(),

            nn.Linear(256,1),
            nn.Sigmoid() #[1,-1]
        )
    
    def forward(self, img):
        return self.model(img)

In [9]:
GAN_Loss = nn.BCELoss()
generator = Generator()
g_optimizer = optim.Adam(generator.parameters(),lr=0.0002, betas=(0.5,0.999))

discriminator = Discriminator()
d_optimizer = optim.Adam(discriminator.parameters(),lr=0.0002, betas=(0.5,0.999))

In [10]:
import torch
 
if torch.cuda.is_available():
    device = torch.device("cuda")
else :
    device = torch.device("cpu")

print(device) 

cuda


In [11]:
generator = generator.to(device)
discriminator = discriminator.to(device)

In [ ]:
# training the GAN
def trainer(generator,discriminator,DataLoader,epochs = 10):

    for epoch in range(epochs):
        for i,img in enumerate(dataLoader):
            real_img = img.to(device)
            batch_size = real_img.size(0)

            #creating labels
            real_labels = torch.ones(batch_size,1).to(device)
            fake_labels = torch.zeros(batch_size,1).to(device)

            #Training the discriminator
            d_optimizer.zero_grad()
            
            fake_img = generator(torch.randn(batch_size,100).to(device))

            fake_loss = GAN_Loss(discriminator(fake_img.detach()),fake_labels)
            real_loss = GAN_Loss(discriminator(real_img),real_labels)

            d_loss = fake_loss + real_loss/2

            d_loss.backward()
            d_optimizer.step()

            # Train Generator
            g_optimizer.zero_grad()
            g_loss = GAN_Loss(discriminator(fake_img),real_labels)

            g_loss.backward()
            g_optimizer.step()

            if i % 50 ==0:
                print(f"|| epoch:{epoch+1}/{epochs} || batch {batch}/{batch_size}|| Giscriminator loss:{d_loss} || Generator loss : {g_loss} ||")
        
        save_generated_img(generator, epoch, device)
        

In [13]:
import matplotlib.pyplot as plt
import torchvision
def save_generated_img(generator, epoch, device, num_imgs=8):
    z=torch.randn(num_imgs,100).to(device)
    generated_img = generator(z).detach().cpu()

    from torchvision.utils import make_grid
    grid = make_grid(generated_img, nrow=4, normalize=True)

    plt.imshow(np.transpose(grid, (1,2,0))) #plt expect (height,weight,channel) 
    plt.title(f"epoch:{epoch}")
    plt.axis("off")
    plt.show()

In [14]:
trainer(generator,discriminator,dataLoader,epochs=5)

|| epoch:1/5 || Giscriminator loss:1.0336157083511353 || Generator loss : 0.7139943838119507 ||
|| epoch:1/5 || Giscriminator loss:0.6749903559684753 || Generator loss : 1.1096673011779785 ||
|| epoch:1/5 || Giscriminator loss:0.9600322842597961 || Generator loss : 0.6621387004852295 ||
|| epoch:1/5 || Giscriminator loss:0.3101874887943268 || Generator loss : 1.495120644569397 ||
|| epoch:1/5 || Giscriminator loss:0.6589851379394531 || Generator loss : 1.8096306324005127 ||
|| epoch:1/5 || Giscriminator loss:0.26120609045028687 || Generator loss : 1.7058496475219727 ||
|| epoch:1/5 || Giscriminator loss:0.4366735816001892 || Generator loss : 1.6069879531860352 ||
|| epoch:1/5 || Giscriminator loss:0.2968139350414276 || Generator loss : 2.168043851852417 ||
|| epoch:1/5 || Giscriminator loss:0.08317406475543976 || Generator loss : 3.433241844177246 ||
|| epoch:1/5 || Giscriminator loss:0.16060100495815277 || Generator loss : 2.76454758644104 ||
|| epoch:1/5 || Giscriminator loss:0.14815

UnidentifiedImageError: cannot identify image file './img_align_celeba\\DCGAN.ipynb'